# Module 05: Secure Boot — Lab

This lab simulates a secure boot chain and explores common vulnerabilities.

**Objectives:**
1. Simulate a chain of trust with hash verification
2. Demonstrate measured boot PCR extend operation
3. Simulate a downgrade attack and its detection
4. Visualize cold boot DRAM data remanence

In [ ]:
import hashlib
import struct
import numpy as np
import matplotlib.pyplot as plt

print("=" * 60)
print("CHAIN OF TRUST SIMULATION")
print("=" * 60)

# Simulate boot stages with hash verification
class SecureBoot:
    def __init__(self, signing_key="root-secret-key-1234"):
        self.signing_key = signing_key.encode()
        self.trust_chain = []
        
    def sign(self, data):
        """Simulate signing data with HMAC"""
        return hashlib.sha256(data + self.signing_key).hexdigest()
    
    def verify(self, data, signature):
        """Verify signature matches data"""
        return self.sign(data) == signature
    
    def boot_stage(self, name, code, expected_sig=None):
        """Execute a boot stage with verification"""
        code_hash = hashlib.sha256(code).hexdigest()
        print(f"\n[BOOT] {name}: hash={code_hash[:16]}...")
        
        if expected_sig is not None:
            if self.verify(code, expected_sig):
                print(f"  [PASS] Signature verified for {name}")
                self.trust_chain.append(name)
                return True
            else:
                print(f"  [FAIL] Signature INVALID for {name} - BOOT HALTED!")
                return False
        else:
            print(f"  [INFO] Root stage - no verification needed")
            self.trust_chain.append(name)
            return True

# Create secure boot instance
boot = SecureBoot()

# Stage 0: BootROM (immutable root)
bootrom_code = b"BOOTROM_v1.0: Initialize hardware, verify bootloader"
boot.boot_stage("BootROM (Root of Trust)", bootrom_code)

# Stage 1: First-stage bootloader (signed)
fsbl_code = b"FSBL_v2.1: Init DRAM, load U-Boot"
fsbl_sig = boot.sign(fsbl_code)
boot.boot_stage("First-Stage Bootloader", fsbl_code, fsbl_sig)

# Stage 2: Second-stage bootloader (signed)
ssbl_code = b"U-Boot_2023.04: Load Linux kernel"
ssbl_sig = boot.sign(ssbl_code)
boot.boot_stage("Second-Stage Bootloader (U-Boot)", ssbl_code, ssbl_sig)

# Stage 3: OS Kernel (signed)
kernel_code = b"Linux_6.1.0: Boot operating system"
kernel_sig = boot.sign(kernel_code)
boot.boot_stage("OS Kernel (Linux)", kernel_code, kernel_sig)

print(f"\nTrust chain established: {' -> '.join(boot.trust_chain)}")

# Simulate attack: tampered bootloader
print("\n" + "=" * 60)
print("SIMULATING TAMPERED BOOTLOADER ATTACK")
print("=" * 60)
tampered_code = b"MALWARE_FSBL: Execute payload"
boot2 = SecureBoot()
boot2.boot_stage("BootROM (Root of Trust)", bootrom_code)
boot2.boot_stage("First-Stage Bootloader (TAMPERED)", tampered_code, fsbl_sig)

In [ ]:
# Measured Boot PCR Simulation
print("=" * 60)
print("MEASURED BOOT - PCR EXTEND SIMULATION")
print("=" * 60)

class MeasuredBoot:
    def __init__(self, pcr_count=8):
        self.pcrs = [0] * pcr_count  # Initialize all PCRs to 0
        self.log = []
    
    def extend(self, pcr_index, data):
        """PCR extend: PCR_new = SHA256(PCR_old || data)"""
        old_val = self.pcrs[pcr_index]
        old_bytes = old_val.to_bytes(32, 'big')
        data_hash = hashlib.sha256(data).digest()
        new_val = int.from_bytes(
            hashlib.sha256(old_bytes + data_hash).digest(), 'big'
        )
        self.pcrs[pcr_index] = new_val
        self.log.append({
            'pcr': pcr_index,
            'data_hash': data_hash.hex()[:16],
            'new_pcr': hex(new_val)[:18]
        })
        return new_val

# Simulate measured boot sequence
mb = MeasuredBoot()

# Stage measurements
stages = [
    (0, b"BootROM: hardware init"),
    (0, b"FSBL: DRAM init"),
    (1, b"U-Boot: bootloader"),
    (1, b"Linux kernel"),
    (2, b"init system"),
    (2, b"userspace apps"),
]

for pcr_idx, data in stages:
    mb.extend(pcr_idx, data)
    print(f"PCR[{pcr_idx}] extended with: {data.decode()}")

print(f"\nFinal PCR values:")
for i, val in enumerate(mb.pcrs[:4]):
    print(f"  PCR[{i}] = {hex(val)[:20]}...")

# Simulate replay attack: same boot but different measurement order
print("\n" + "=" * 60)
print("DETECTING REPLAY/TAMPER VIA PCR MISMATCH")
print("=" * 60)

mb_tampered = MeasuredBoot()
tampered_stages = [
    (0, b"BootROM: hardware init"),
    (0, b"MALWARE: injected payload"),  # Tampered!
    (1, b"U-Boot: bootloader"),
    (1, b"Linux kernel"),
    (2, b"init system"),
    (2, b"userspace apps"),
]

for pcr_idx, data in tampered_stages:
    mb_tampered.extend(pcr_idx, data)

print(f"\nOriginal PCR[0]: {hex(mb.pcrs[0])[:20]}...")
print(f"Tampered PCR[0]: {hex(mb_tampered.pcrs[0])[:20]}...")
print(f"PCR match: {'SAME (TAMPER DETECTED!)' if mb.pcrs[0] != mb_tampered.pcrs[0] else 'MATCH'}")

In [ ]:
# Cold Boot Attack: DRAM Data Remanence Simulation
print("=" * 60)
print("COLD BOOT ATTACK - DRAM DATA REMANENCE")
print("=" * 60)

# Simulate DRAM cell data decay over time
np.random.seed(42)
dram_size = 1024  # Simulated DRAM cells
original_data = np.random.randint(0, 2, dram_size)  # Original data

# Data retention probability over time (exponential decay)
time_points = np.linspace(0, 300, 300)  # seconds
temperatures = {
    'Room (25°C)': 0.01,      # Fast decay
    'Cool (-20°C)': 0.003,    # Slower decay
    'Frozen (-50°C)': 0.001,  # Very slow decay
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Data retention over time
for temp_name, decay_rate in temperatures.items():
    retention = np.exp(-decay_rate * time_points)
    axes[0].plot(time_points, retention, label=temp_name, linewidth=2)

axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Data Retention Probability')
axes[0].set_title('DRAM Data Decay at Different Temperatures')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1.05])

# Plot 2: Recoverable data bits over time
for temp_name, decay_rate in temperatures.items():
    retention = np.exp(-decay_rate * time_points)
    recoverable = dram_size * retention
    axes[1].plot(time_points, recoverable, label=temp_name, linewidth=2)

axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Recoverable Bits')
axes[1].set_title('DRAM Bits Recoverable After Power-Off')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key Insight: At -50°C, significant data remains recoverable")
print("for minutes to hours after power-off.")
print("\nMitigation: DRAM encryption, memory zeroization on power-off,")
print("or TEE-based key destruction on tamper detection.")